# Session 1b: Second-Seed Training + 2-Seed Ensemble

This notebook trains a **second independent run** (different random seed) of all 5 uncertainty strategies, on the **same 100k image subsample** as your original Session 1, then averages predictions across both seeds — a lightweight approximation of the paper's own approach (they average 30 checkpoints across 3 runs; we approximate this with 2 runs averaged).

**Why this, not just more data:** the mismatch between your winning strategies and the paper's is primarily a variance problem — a single run's AUC has real randomness from weight initialization and batch order. Averaging two independent runs directly targets that variance, which raw data scale alone doesn't fix.

### Before running — in the Kaggle UI:
1. **Enable GPU**: Settings → Accelerator → GPU T4 x2
2. **Add the CheXpert dataset**: + Add Input → search "chexpert"
3. **Add your Session 1 (100k) notebook's output**: + Add Input → search for it under "Your Work" → add it. This gives access to your seed-1 checkpoints.


### Locate dataset and seed-1 checkpoints

In [ ]:
import os, glob

DATA_ROOT = None
for root, dirs, filenames in os.walk('/kaggle/input'):
    if 'train.csv' in filenames:
        DATA_ROOT = root
        break
if DATA_ROOT is None:
    raise FileNotFoundError("CheXpert dataset not found under /kaggle/input — add it via '+ Add Input'.")
print(f"Found dataset at: {DATA_ROOT}")

all_checkpoints = sorted(glob.glob('/kaggle/input/**/*.pth', recursive=True))
print(f"Found {len(all_checkpoints)} existing checkpoint files (should be your 15 seed-1 checkpoints):")
for c in all_checkpoints:
    print(" ", c)

def latest_checkpoint_for(prefix):
    matches = [c for c in all_checkpoints if os.path.basename(c).startswith(prefix)]
    matches.sort(key=lambda p: int(''.join(filter(str.isdigit, os.path.basename(p).split('_e')[-1]))))
    return matches[-1]

seed1_checkpoint_paths = {
    'U-Zeros': latest_checkpoint_for('zeros_e'),
    'U-Ones': latest_checkpoint_for('ones_e'),
    'U-Ignore': latest_checkpoint_for('ignore_e'),
    'U-SelfTrained': latest_checkpoint_for('selftrained_e'),
    'U-MultiClass': latest_checkpoint_for('multiclass_e'),
}
print("\nSeed-1 checkpoints:", seed1_checkpoint_paths)


### Rebuild pipeline (dataset class, model builders, loss functions, evaluation)

Identical to Session 1 — needed since this is a fresh session. **Critically, `raw_train_df` uses the same `random_state=42`**, so it's the exact same 100k images as before — only the model's random seed changes.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from sklearn.metrics import roc_auc_score
from PIL import Image

PATHOLOGIES = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
BATCH_SIZE = 16
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

train_transform = transforms.Compose([
    transforms.Resize((320, 320)), transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((320, 320)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ImageLabelDataset(Dataset):
    def __init__(self, df, data_root, pathologies, transform, label_dtype=torch.float32):
        self.df = df.reset_index(drop=True)
        self.data_root = data_root
        self.pathologies = pathologies
        self.transform = transform
        self.label_dtype = label_dtype
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        relative_path = row['Path'].replace('CheXpert-v1.0-small/', '')
        img_path = f"{self.data_root}/{relative_path}"
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        np_dtype = np.int64 if self.label_dtype == torch.long else np.float32
        labels = torch.tensor(row[self.pathologies].values.astype(np_dtype), dtype=self.label_dtype)
        return image, labels

N_TRAIN_STRATEGY = 100000  # MUST match Session 1 exactly

full_train_df = pd.read_csv(f'{DATA_ROOT}/train.csv')
full_train_df = full_train_df[full_train_df['Frontal/Lateral'] == 'Frontal'].reset_index(drop=True)
for p in PATHOLOGIES:
    full_train_df[p] = full_train_df[p].fillna(0)

raw_train_df = full_train_df.sample(n=min(N_TRAIN_STRATEGY, len(full_train_df)), random_state=42).reset_index(drop=True)
print(f"Same 100k image subsample as Session 1: {len(raw_train_df)} images")

val_df = pd.read_csv(f'{DATA_ROOT}/valid.csv')
val_df = val_df[val_df['Frontal/Lateral'] == 'Frontal'].reset_index(drop=True)
for p in PATHOLOGIES:
    val_df[p] = val_df[p].fillna(0)

val_dataset = ImageLabelDataset(val_df, DATA_ROOT, PATHOLOGIES, val_transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Validation size: {len(val_dataset)}")

def build_model(num_classes=len(PATHOLOGIES)):
    model = models.densenet121(weights='DEFAULT')
    model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    return model.to(device)

def build_model_multiclass(num_classes=len(PATHOLOGIES)):
    model = models.densenet121(weights='DEFAULT')
    model.classifier = nn.Linear(model.classifier.in_features, num_classes * 3)
    return model.to(device)

scaler = GradScaler('cuda')
bce_criterion = nn.BCEWithLogitsLoss()

def masked_bce_loss(outputs, labels):
    mask = ~torch.isnan(labels)
    labels_filled = torch.where(mask, labels, torch.zeros_like(labels))
    loss_elem = F.binary_cross_entropy_with_logits(outputs, labels_filled, reduction='none')
    return (loss_elem * mask.float()).sum() / mask.float().sum().clamp(min=1.0)

def train_one_epoch_generic(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            outputs = model(images)
            loss = loss_fn(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def train_one_epoch_multiclass(model, loader, optimizer, device, num_pathologies):
    model.train()
    total_loss = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            outputs = model(images).view(-1, num_pathologies, 3).permute(0, 2, 1)
            loss = F.cross_entropy(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, device, pathologies):
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = torch.sigmoid(model(images))
            all_preds.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())
    all_preds = np.concatenate(all_preds); all_labels = np.concatenate(all_labels)
    aucs = {}
    for i, p in enumerate(pathologies):
        try:
            aucs[p] = roc_auc_score(all_labels[:, i], all_preds[:, i])
        except ValueError:
            aucs[p] = float('nan')
    return aucs

def evaluate_multiclass(model, loader, device, pathologies):
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images).view(-1, len(pathologies), 3)
            logit_neg, logit_pos = outputs[:, :, 0], outputs[:, :, 1]
            p_pos = torch.softmax(torch.stack([logit_neg, logit_pos], dim=-1), dim=-1)[..., 1]
            all_probs.append(p_pos.cpu().numpy())
            all_labels.append(labels.numpy())
    all_probs = np.concatenate(all_probs); all_labels = np.concatenate(all_labels)
    aucs = {}
    for i, p in enumerate(pathologies):
        try:
            aucs[p] = roc_auc_score(all_labels[:, i], all_probs[:, i])
        except ValueError:
            aucs[p] = float('nan')
    return aucs

os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
os.makedirs('/kaggle/working/results', exist_ok=True)
NUM_EPOCHS = 3
SEED2 = 123


### Train second-seed U-Zeros and U-Ones

`torch.manual_seed(SEED2)` before model construction changes weight initialization; the DataLoader's `shuffle=True` combined with the new seed also changes batch ordering — together these produce a genuinely independent second run on the same data.

In [ ]:
def run_training_seeded(strategy_name, dataset, num_epochs, loss_fn, save_prefix, seed):
    torch.manual_seed(seed)
    model = build_model()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    for epoch in range(num_epochs):
        train_loss = train_one_epoch_generic(model, loader, optimizer, loss_fn, device)
        aucs = evaluate(model, val_loader, device, PATHOLOGIES)
        print(f"[{strategy_name} seed{seed}] Epoch {epoch+1}/{num_epochs} — Loss: {train_loss:.4f}")
        for p, a in aucs.items():
            print(f"    {p}: {a:.4f}")
        torch.save(model.state_dict(), f'/kaggle/working/checkpoints/{save_prefix}_seed2_e{epoch+1}.pth')
    final_aucs = evaluate(model, val_loader, device, PATHOLOGIES)
    return model, final_aucs

df_zeros = raw_train_df.copy()
for p in PATHOLOGIES:
    df_zeros[p] = df_zeros[p].replace(-1, 0)
dataset_zeros = ImageLabelDataset(df_zeros, DATA_ROOT, PATHOLOGIES, train_transform)
model_zeros_s2, aucs_zeros_s2 = run_training_seeded('U-Zeros', dataset_zeros, NUM_EPOCHS, bce_criterion, 'zeros', SEED2)


In [ ]:
df_ones = raw_train_df.copy()
for p in PATHOLOGIES:
    df_ones[p] = df_ones[p].replace(-1, 1)
dataset_ones = ImageLabelDataset(df_ones, DATA_ROOT, PATHOLOGIES, train_transform)
model_ones_s2, aucs_ones_s2 = run_training_seeded('U-Ones', dataset_ones, NUM_EPOCHS, bce_criterion, 'ones', SEED2)


### Train second-seed U-Ignore

In [ ]:
df_ignore = raw_train_df.copy()
for p in PATHOLOGIES:
    df_ignore[p] = df_ignore[p].replace(-1, np.nan)
dataset_ignore = ImageLabelDataset(df_ignore, DATA_ROOT, PATHOLOGIES, train_transform)
model_ignore_s2, aucs_ignore_s2 = run_training_seeded('U-Ignore', dataset_ignore, NUM_EPOCHS, masked_bce_loss, 'ignore', SEED2)


### Train second-seed U-SelfTrained

A genuinely independent second run means bootstrapping from **this run's own** U-Ignore model (`model_ignore_s2`), not reusing seed 1's — matching how the paper's independent runs would each have their own internal bootstrap.

In [ ]:
uncertain_mask_df = (raw_train_df[PATHOLOGIES] == -1)
inference_df = raw_train_df.copy()
for p in PATHOLOGIES:
    inference_df[p] = inference_df[p].replace(-1, 0)
inference_dataset = ImageLabelDataset(inference_df, DATA_ROOT, PATHOLOGIES, val_transform)
inference_loader = DataLoader(inference_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

model_ignore_s2.eval()
all_soft_preds = []
with torch.no_grad():
    for images, _ in inference_loader:
        images = images.to(device)
        preds = torch.sigmoid(model_ignore_s2(images)).cpu().numpy()
        all_soft_preds.append(preds)
all_soft_preds = np.concatenate(all_soft_preds)

df_selftrained = raw_train_df.copy()
for i, p in enumerate(PATHOLOGIES):
    col_mask = uncertain_mask_df[p].values
    df_selftrained.loc[col_mask, p] = all_soft_preds[col_mask, i]

dataset_selftrained = ImageLabelDataset(df_selftrained, DATA_ROOT, PATHOLOGIES, train_transform)
model_selftrained_s2, aucs_selftrained_s2 = run_training_seeded(
    'U-SelfTrained', dataset_selftrained, NUM_EPOCHS, bce_criterion, 'selftrained', SEED2
)


### Train second-seed U-MultiClass

In [ ]:
df_multiclass = raw_train_df.copy()
for p in PATHOLOGIES:
    df_multiclass[p] = df_multiclass[p].replace(-1, 2).astype(int)
dataset_multiclass = ImageLabelDataset(df_multiclass, DATA_ROOT, PATHOLOGIES, train_transform, label_dtype=torch.long)
loader_multiclass = DataLoader(dataset_multiclass, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

torch.manual_seed(SEED2)
model_multiclass_s2 = build_model_multiclass()
optimizer_mc = optim.Adam(model_multiclass_s2.parameters(), lr=1e-4)

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch_multiclass(model_multiclass_s2, loader_multiclass, optimizer_mc, device, len(PATHOLOGIES))
    aucs = evaluate_multiclass(model_multiclass_s2, val_loader, device, PATHOLOGIES)
    print(f"[U-MultiClass seed{SEED2}] Epoch {epoch+1}/{NUM_EPOCHS} — Loss: {train_loss:.4f}")
    for p, a in aucs.items():
        print(f"    {p}: {a:.4f}")
    torch.save(model_multiclass_s2.state_dict(), f'/kaggle/working/checkpoints/multiclass_seed2_e{epoch+1}.pth')

aucs_multiclass_s2 = evaluate_multiclass(model_multiclass_s2, val_loader, device, PATHOLOGIES)


### Build 2-seed ensembles and reload seed-1 models

`SeedEnsembleModel` wraps both seeds' models for a strategy and averages their predicted probabilities — this is the actual paper-style ensembling mechanism (just 2 runs instead of 3, and per-epoch checkpoints instead of the paper's 10-per-run).

In [ ]:
# Reload seed-1 models
model_zeros_s1 = build_model(); model_zeros_s1.load_state_dict(torch.load(seed1_checkpoint_paths['U-Zeros'], map_location=device)); model_zeros_s1.eval()
model_ones_s1 = build_model(); model_ones_s1.load_state_dict(torch.load(seed1_checkpoint_paths['U-Ones'], map_location=device)); model_ones_s1.eval()
model_ignore_s1 = build_model(); model_ignore_s1.load_state_dict(torch.load(seed1_checkpoint_paths['U-Ignore'], map_location=device)); model_ignore_s1.eval()
model_selftrained_s1 = build_model(); model_selftrained_s1.load_state_dict(torch.load(seed1_checkpoint_paths['U-SelfTrained'], map_location=device)); model_selftrained_s1.eval()
model_multiclass_s1 = build_model_multiclass(); model_multiclass_s1.load_state_dict(torch.load(seed1_checkpoint_paths['U-MultiClass'], map_location=device)); model_multiclass_s1.eval()

class SeedEnsembleModel(nn.Module):
    """Averages predicted probabilities across 2 independently-trained seeds of the same strategy,
    returning logits (via inverse-sigmoid) so it's a drop-in replacement anywhere a single model is expected."""
    def __init__(self, model_a, model_b, is_multiclass=False, num_pathologies=len(PATHOLOGIES)):
        super().__init__()
        self.model_a = model_a
        self.model_b = model_b
        self.is_multiclass = is_multiclass
        self.num_pathologies = num_pathologies

    def _to_probs(self, model, x):
        out = model(x)
        if self.is_multiclass:
            out = out.view(-1, self.num_pathologies, 3)
            logit_neg, logit_pos = out[:, :, 0], out[:, :, 1]
            return torch.softmax(torch.stack([logit_neg, logit_pos], dim=-1), dim=-1)[..., 1]
        return torch.sigmoid(out)

    def forward(self, x):
        p_a = self._to_probs(self.model_a, x)
        p_b = self._to_probs(self.model_b, x)
        p_avg = ((p_a + p_b) / 2).clamp(1e-6, 1 - 1e-6)
        return torch.log(p_avg / (1 - p_avg))  # back to logit space

ensemble_zeros = SeedEnsembleModel(model_zeros_s1, model_zeros_s2).to(device)
ensemble_ones = SeedEnsembleModel(model_ones_s1, model_ones_s2).to(device)
ensemble_ignore = SeedEnsembleModel(model_ignore_s1, model_ignore_s2).to(device)
ensemble_selftrained = SeedEnsembleModel(model_selftrained_s1, model_selftrained_s2).to(device)
ensemble_multiclass = SeedEnsembleModel(model_multiclass_s1, model_multiclass_s2, is_multiclass=True).to(device)

ensembled_strategy_models = {
    'U-Zeros': ensemble_zeros, 'U-Ones': ensemble_ones, 'U-Ignore': ensemble_ignore,
    'U-SelfTrained': ensemble_selftrained, 'U-MultiClass': ensemble_multiclass,
}

ensembled_aucs = {name: evaluate(model, val_loader, device, PATHOLOGIES) for name, model in ensembled_strategy_models.items()}

comparison_df_ensembled = pd.DataFrame(ensembled_aucs).T[PATHOLOGIES]
print("=== 2-Seed Ensembled AUC by Strategy and Pathology ===")
print(comparison_df_ensembled.round(4).to_string())

best_strategy_per_pathology_ensembled = comparison_df_ensembled.idxmax(axis=0)
print("\n=== Best Strategy per Pathology (2-seed ensembled) ===")
for p in PATHOLOGIES:
    print(f"  {p}: {best_strategy_per_pathology_ensembled[p]} (AUC = {comparison_df_ensembled.loc[best_strategy_per_pathology_ensembled[p], p]:.4f})")

comparison_df_ensembled.to_csv('/kaggle/working/results/strategy_comparison_2seed.csv')


### Build the final 2-seed composite model

In [ ]:
best_per_pathology_ensembled = {
    p: (best_strategy_per_pathology_ensembled[p], ensembled_strategy_models[best_strategy_per_pathology_ensembled[p]])
    for p in PATHOLOGIES
}

class CompositeBestModel(nn.Module):
    def __init__(self, best_per_pathology, pathologies):
        super().__init__()
        self.pathologies = pathologies
        self.best_per_pathology = best_per_pathology
        self.unique_models = nn.ModuleDict()
        for i, (p, (strat, m)) in enumerate(best_per_pathology.items()):
            key = f"{strat}_{i}"
            self.unique_models[key] = m
        self._strat_to_key = {}
        for i, (p, (strat, m)) in enumerate(best_per_pathology.items()):
            self._strat_to_key[p] = f"{strat}_{i}"

    def forward(self, x):
        batch = x.size(0)
        final_logits = torch.zeros(batch, len(self.pathologies), device=x.device)
        cache = {}
        for i, p in enumerate(self.pathologies):
            key = self._strat_to_key[p]
            if key not in cache:
                cache[key] = self.unique_models[key](x)
            final_logits[:, i] = cache[key][:, i]
        return final_logits

composite_model_2seed = CompositeBestModel(best_per_pathology_ensembled, PATHOLOGIES).to(device)
composite_model_2seed.eval()

composite_aucs_2seed = evaluate(composite_model_2seed, val_loader, device, PATHOLOGIES)
print("=== Final 2-Seed Composite Model AUCs ===")
for p, a in composite_aucs_2seed.items():
    print(f"  {p}: {a:.4f}")

comparison_df_ensembled.loc['Composite (2-seed final)'] = pd.Series(composite_aucs_2seed)
comparison_df_ensembled.to_csv('/kaggle/working/results/strategy_comparison_2seed.csv')
print(sorted(os.listdir('/kaggle/working/checkpoints')))


## ✅ Session 1b Complete — Save Your Progress

Click **Save Version → Save & Run All (Commit)**.

This commits 10 new second-seed checkpoints plus `strategy_comparison_2seed.csv`. For Session 2, add **both** this notebook's output and your original Session 1's output as Inputs, so all 20 checkpoints (10 seed-1 + 10 seed-2) are available to rebuild the `SeedEnsembleModel`s and the final composite.